In [0]:
from pyspark.sql import functions as F

# Introduction
First I load the data, drop duplicates and look through the dataset: what the rows look like, whether anything is inconsistent, what the formats are, and which columns are empty and how to deal with them.

In [0]:
df = spark.read.table("default.game_data")
total = df.count()
distinct = df.distinct().count()
print(f"Total rows = {total}, distinct = {distinct}, duplicates = {total - distinct}")

# drop duplicates
df = df.dropDuplicates()
raw_df = df
print(f"Rows after dropDuplicates = {raw_df.count()}")
display(raw_df)

Total rows = 183038, distinct = 91517, duplicates = 91521
Rows after dropDuplicates = 91517


The UserScore column turns out to be empty. I fill it in myself from the positive and negative review counts, scaled to 0–10, so there's some kind of user rating to work with.

In [0]:
df_cast = df \
  .withColumn("Positive_int", F.expr("try_cast(Positive AS INT)")) \
  .withColumn("Negative_int", F.expr("try_cast(Negative AS INT)"))

# 2) Count rows where try_cast returned NULL, i.e. broken records
invalid_count = df_cast.filter(
    F.col("Positive_int").isNull() | F.col("Negative_int").isNull()
).count()

print(f"Rows with invalid Positive/Negative: {invalid_count}")

# 3) Keep only valid rows and rename back to Positive/Negative
df = (
  df_cast
    .filter(F.col("Positive_int").isNotNull() & F.col("Negative_int").isNotNull())
    .drop("Positive", "Negative")
    .withColumnRenamed("Positive_int", "Positive")
    .withColumnRenamed("Negative_int", "Negative")
)

# 4) Compute a new 0–10 UserScore, rounded to 2 decimals
df = df.withColumn(
    "UserScore",
    F.round(
      F.when(
        (F.col("Positive") + F.col("Negative")) > 0,
        (F.col("Positive") / (F.col("Positive") + F.col("Negative"))) * 10
      ).otherwise(0),
      2
    )
)

# 5) Look at the first few rows to check it
display(df.select("AppID", "Positive", "Negative", "UserScore").limit(20))

Rows with invalid Positive/Negative: 25


AppID,Positive,Negative,UserScore
1818320,15,7,6.82
541150,0,0,0.0
1986620,0,0,0.0
991740,35,37,4.86
690250,8,7,5.33
403970,1318,472,7.36
613620,22,12,6.47
711350,0,3,0.0
781130,80,21,7.92
1688010,0,0,0.0


This has an obvious problem: a game with a single positive review gets a 10, which doesn't mean much. So games need a minimum number of reviews before their score counts, and I use the median total review count as that cut-off.

In [0]:
df = df.withColumn("totalReviews", F.col("Positive") + F.col("Negative"))


median_val = df.stat.approxQuantile("totalReviews", [0.5], 0.01)[0]
print(f"Median total reviews = {median_val}")

df_filtered = df.filter(F.col("totalReviews") >= median_val)

print(f"Before filter: {df.count()} games")
print(f"After filter: {df_filtered.count()} games")

display(df_filtered.select("AppID","Name","totalReviews","Positive","Negative","UserScore").limit(20))

Median total reviews = 7.0
Before filter: 91492 games
After filter: 47780 games


AppID,Name,totalReviews,Positive,Negative,UserScore
1818320,いせかいあどべんちゃー,22,15,7,6.82
991740,Era Of Newborns,72,35,37,4.86
690250,Judge Dredd 95,15,8,7,5.33
403970,The Dwarves,1790,1318,472,7.36
613620,Hoverloop,34,22,12,6.47
781130,Creatura,101,80,21,7.92
476370,MAGATAMA Earrings,19,13,6,6.84
434250,Masked Shooters 2,1599,1133,466,7.09
1834620,The Shore VR,42,16,26,3.81
229810,A Fistful of Gun,461,295,166,6.4


That roughly halves the dataset (91,492 → 47,780 games), which is the point: what's left has scores worth comparing. Next I compare scores with price, by publisher and developer, to see who makes the best games for the money. Basically, looking for bang for the buck.

In [0]:
from pyspark.sql.functions import col

# df_filtered comes from the previous cell:
#   df_filtered = df.filter(F.col("totalReviews") >= median_val)

# 1) Cast Price to double and count how many values fail
df_price = df_filtered \
  .withColumn("Price_double", F.expr("try_cast(Price AS DOUBLE)"))

bad_price_count = df_price.filter(col("Price_double").isNull()).count()
print(f"Invalid Price values: {bad_price_count}")

# 2) Drop rows with an invalid Price and keep only the numeric one
df_clean = df_price \
  .filter(col("Price_double").isNotNull()) \
  .drop("Price") \
  .withColumnRenamed("Price_double", "Price")

# 3) Make sure UserScore is a DOUBLE too
df = df_clean.withColumn("UserScore", col("UserScore").cast("double"))

# df now has numeric Price and UserScore, ready for the analyses below
print("Schema check:") 
df.printSchema()

Invalid Price values: 0
Schema check:
root
 |-- AppID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- ReleaseDate: string (nullable = true)
 |-- EstimatedOwners: string (nullable = true)
 |-- PeakCCU: string (nullable = true)
 |-- RequiredAge: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- DLCCount: string (nullable = true)
 |-- AboutTheGame: string (nullable = true)
 |-- SupportedLanguages: string (nullable = true)
 |-- FullAudioLanguages: string (nullable = true)
 |-- Reviews: string (nullable = true)
 |-- HeaderImage: string (nullable = true)
 |-- Website: string (nullable = true)
 |-- SupportURL: string (nullable = true)
 |-- SupportEmail: string (nullable = true)
 |-- Windows: string (nullable = true)
 |-- Mac: string (nullable = true)
 |-- Linux: string (nullable = true)
 |-- MetacriticScore: string (nullable = true)
 |-- MetacriticURL: string (nullable = true)
 |-- UserScore: double (nullable = true)
 |-- ScoreRank: string (nullable = tr

# Analysis 0: Warming up with Spark

This looks at publishers and how well rated their games are relative to price ("score per euro"). Publishers are split out of the list column, and each game is flagged as self-published (the publisher is also one of the developers) or published for someone else. Average price and score per publisher show who gives the most quality for the money. Only publishers with at least 3 games are kept.

Useful for players looking for cheap but good games, and for publishers comparing their pricing with the competition.

Result: the ratio heavily favours very cheap games, so the top of the list is publishers whose games cost well under 1 on average (e.g. Paradise Project, 24 games at 0.17 with a 7.63 score). A minimum price filter would make the ranking more meaningful for regular-priced games.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import split, explode, trim, col, when, count, sum, avg

# 1) Explode publishers into separate rows
df_pub = (
  df
    .withColumn("Publisher", explode(split(col("Publishers"), ";")))
    .withColumn("Publisher", trim(col("Publisher")))
)

# 2) Flag whether the publisher is also one of the developers
df_pub = df_pub.withColumn(
  "self_published",
  when(
    F.array_contains(split(col("Developers"), ";"), col("Publisher")),
    True
  ).otherwise(False)
)

# 3) Group and aggregate
pub_stats = (
  df_pub
    .groupBy("Publisher")
    .agg(
      count("*").alias("total_games"),
      sum(when(col("self_published"), 1).otherwise(0)).alias("self_published_games"),
      sum(when(~col("self_published"), 1).otherwise(0)).alias("external_games"),
      F.round(avg("Price"), 2).alias("avg_price"),
      F.round(avg("UserScore"), 2).alias("avg_score")
    )
    .withColumn(
      "score_per_euro",
      F.when(col("avg_price") > 0, F.round(col("avg_score") / col("avg_price"), 2))
      .otherwise(None)
    )
    .filter(col("total_games") >= 3)
    .orderBy(col("score_per_euro").desc())
)

display(pub_stats)


Publisher,total_games,self_published_games,external_games,avg_price,avg_score,score_per_euro
Paradise Project,24,0,24,0.17,7.63,44.88
none,8,5,3,0.12,5.33,44.42
ToastieLabs,4,4,0,0.25,7.66,30.64
Mad Labyrinth Studios LLC,3,3,0,0.25,5.97,23.88
Triority Interactive Novels,6,6,0,0.33,7.09,21.48
Ordinary Pine Studio,3,0,3,0.49,9.72,19.84
迷糊的安安,6,1,5,0.5,9.51,19.02
FoxSox Games,3,3,0,0.53,9.29,17.53
Virtual World Arcade,4,3,1,0.5,8.6,17.2
Self Published,4,0,4,0.5,8.6,17.2


# Analysis 0.5: Genres

Games can have several genres, so I split them out and compute, per genre, the number of games, the average user score and the average price. Sorting by average score shows which genres players rate highest and what a game in each genre typically costs. Only genres with at least 5 games are kept.

Result: Casual (7.69), Indie (7.66) and Adventure (7.65) are at the top, but most genres sit between 7.0 and 7.7, so the differences are small. Massively Multiplayer (6.45) is the lowest of the big genres.

In [0]:
# 1) Explode the semicolon-separated genres into separate rows
df_genre = (
    df
      .withColumn("Genre", explode(split(col("Genres"), ";")))
      .withColumn("Genre", trim(col("Genre")))
)

# 2) Group and aggregate
df_genre_stats = (
    df_genre
      .groupBy("Genre")
      .agg(
         count("*").alias("num_games"),              # games per genre
         avg("UserScore").alias("avg_user_score"),   # average score (0–10)
         avg("Price").alias("avg_price")             # average price
      )
      .filter(col("num_games") >= 5)                # at least 5 games so the averages mean something
      .orderBy(col("avg_user_score").desc())       # best-rated genres first
)

display(df_genre_stats)

Genre,num_games,avg_user_score,avg_price
Casual,17805,7.68900365065992,6.0741477113169084
Indie,33972,7.656675497468488,7.289073060167667
Game Development,120,7.65575,25.526333333333337
Adventure,19249,7.651744506208108,8.965257935477336
Photo Editing,76,7.582763157894736,23.65144736842106
RPG,8884,7.578994822152186,10.296983340837327
Action,20282,7.513293560792814,8.821242481017853
Early Access,5438,7.467927546892245,10.033177638837698
Web Publishing,76,7.463157894736843,21.182236842105265
Strategy,9669,7.423428482780017,9.438344192780917


# Analysis 1: Pearson correlation between price and score

The old question: do good games have to be expensive? A value near 1 would mean a higher price goes with a higher score, near -1 the opposite, and near 0 there's no linear relationship.

Result: 0.0822, so there's no meaningful linear relationship between price and average user score. Expensive games aren't necessarily rated better and cheap ones aren't necessarily worse. Price on its own is a poor indicator of how players rate a game.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# 1) Cast price to double
df = df.withColumn("Price", F.col("Price").cast(DoubleType()))

# 2) Pearson correlation
corr = df.stat.corr("UserScore", "Price")
print(f"Pearson corr(Price, UserScore) = {corr:.4f}")


Pearson corr(Price, UserScore) = 0.0822


# Analysis 2: Linux and games

Linux has been getting more popular with gamers: it's open source, more private and, set up properly, often runs better, and Microsoft's own privacy decisions haven't hurt either. The idea here is to find the turning point, the year when the number of games supporting Linux really started to grow.

Result: aggregating by release year and looking at the absolute year-over-year growth in Linux-supported games, the turning point is 2015, with 351 more Linux games than in 2014. About 31% of that year's releases supported Linux. The share itself peaked earlier (38% in 2013) and has been falling since, because the total number of releases grew much faster. 2024 and 2025 are incomplete in the data.

In [0]:
spark.conf.set("spark.sql.ansi.enabled", "false")


from pyspark.sql import functions as F
from pyspark.sql.functions import year, col, desc, coalesce, to_date
from pyspark.sql.window import Window

parsed_df = (
    raw_df
    .withColumn(
        "parsed_dt",
        coalesce(
            to_date(col("ReleaseDate"), "MMM d, yyyy"),   # e.g. "Apr 19, 2018"
            to_date(col("ReleaseDate"), "MMM yyyy")       # e.g. "Dec 2024"
        )
    )
    # drop rows where the date could not be parsed
    .filter(col("parsed_dt").isNotNull())
    # replace the original column with the parsed one
    .drop("ReleaseDate")
    .withColumnRenamed("parsed_dt", "ReleaseDate")
)


# 1) Aggregate by year
yearly = (
    parsed_df
    .withColumn("year", year(col("ReleaseDate")))
    .groupBy("year")
    .agg(
        F.count("*").alias("total_games"),
        F.sum(F.when(col("Linux") == "True", 1).otherwise(0))
         .alias("linux_games")
    )
    .withColumn(
        "linux_pct",
        F.round(col("linux_games") / col("total_games") * 100, 2)
    )
    .orderBy("year")
)

# 2) Year-over-year growth in the absolute number of Linux games
w = Window.orderBy("year")
yearly = (
    yearly
    .withColumn("prev_linux_games", F.lag("linux_games").over(w))
    .withColumn("yoy_linux_growth", col("linux_games") - col("prev_linux_games"))
)

# 3) Show the table
display(yearly)


inflection_row = (
    yearly
    .filter(col("prev_linux_games").isNotNull())   # skip the first year, it has no previous one
    .orderBy(desc("yoy_linux_growth"))
    .limit(1)
    .collect()[0]
)

year_inflect   = inflection_row["year"]
growth_inflect = inflection_row["yoy_linux_growth"]

# 4) Print the conclusion
print(
    f"The turning point for Linux games is {year_inflect}, "
    f"with {growth_inflect} more Linux titles than in {year_inflect - 1}."
)


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


year,total_games,linux_games,linux_pct,prev_linux_games,yoy_linux_growth
1997,2,1,50.0,null,null
1998,1,1,100.0,1,0
1999,3,2,66.67,1,1
2000,2,2,100.0,2,0
2001,4,2,50.0,2,0
2002,1,0,0.0,2,-2
2003,3,1,33.33,0,1
2004,7,6,85.71,1,5
2005,7,2,28.57,6,-4
2006,69,10,14.49,2,8


The turning point for Linux games is 2015, with 351 more Linux titles than in 2014.


# Analysis 3: K-means clustering

Based on an example from the course labs, the idea is to segment games into 4 groups using price, user score, average playtime, peak concurrent players and positive/negative review counts, then use the cluster centres, sizes and per-cluster averages to describe each type of game.

**Status: not run.** On the Databricks Community Edition cluster `VectorAssembler` is blocked (`Py4JSecurityException: ... is not whitelisted`), so this notebook has no clustering results. On a regular Spark setup it should run. The features would also need scaling (e.g. `StandardScaler`) first, since peak players and review counts are orders of magnitude larger than price or score.

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
import matplotlib.pyplot as plt

# 1) Pick the numeric columns and drop nulls
numeric_columns = ["Price", "UserScore", "AvgPlaytimeForever", "PeakCCU", "Positive", "Negative"]
df_clean = raw_df.select(numeric_columns).na.drop()

# 2) Combine them into a feature vector
assembler = VectorAssembler(inputCols=numeric_columns, outputCol="features")
data = assembler.transform(df_clean)

# 3) Train K-means (4 clusters here, easy to change)
kmeans = KMeans(k=4, seed=42, featuresCol="features", predictionCol="cluster")
model = kmeans.fit(data)

# 4) Add cluster labels back to the DataFrame
clustered = model.transform(data)

# 5) Print the cluster centres
print("Cluster centres:")
for idx, center in enumerate(model.clusterCenters()):
    print(f"  Cluster {idx}: {center}")

# 6) Plot the first two features (Price vs UserScore)
clustered_pd = clustered.select("features", "cluster").toPandas()

plt.figure(figsize=(8, 6))
for cluster_id in sorted(clustered_pd["cluster"].unique()):
    pts = clustered_pd[clustered_pd["cluster"] == cluster_id]
    plt.scatter(
        pts["features"].apply(lambda v: v[0]),  # Price
        pts["features"].apply(lambda v: v[1]),  # UserScore
        label=f"Cluster {cluster_id}",
        alpha=0.6
    )

plt.xlabel("Price")
plt.ylabel("UserScore")
plt.title("K-means clustering of games")
plt.legend()
plt.grid(True)
plt.show()


# Analysis 4: Genre popularity over time

Games change constantly. Technology moves forward, and design trends, genres and types of play move with it. Culture matters too: a hit film, politics or changing social habits all leave a mark on which kinds of games are popular.

The goal is to find which genres and categories (modes of play such as single-player or multi-player) dominated each decade, how new trends appear and old ones fade, and what the life cycle of the main genres looks like. That's useful for publishers, developers and industry analysts, and for understanding how gaming culture has changed in general.

Result: single-player is the top category in every decade. Genres moved from mostly action (27% of genre tags in the 2000s) to a more varied catalogue where indie leads from the 2010s on (about 25%). That follows changes in technology and distribution, and players expecting more varied and original games. The 1990s have only a handful of games in the dataset, so that decade's percentages don't say much.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Parse the release date and decade
df = (
    raw_df
    .withColumn("ReleaseDate_parsed",
        F.coalesce(
            F.to_date("ReleaseDate", "MMM d, yyyy"),
            F.to_date("ReleaseDate", "MMM yyyy")
        )
    )
    .filter(F.col("ReleaseDate_parsed").isNotNull())
    .withColumn("year", F.year("ReleaseDate_parsed"))
    .withColumn("decade", (F.col("year") / 10).cast("int") * 10)
)

# 2. Explode categories per decade
cat = (
    df
    .withColumn("category", F.explode(F.split(F.col("Categories"), ";")))
    .withColumn("category", F.trim(F.lower(F.col("category"))))
    .filter(F.col("category") != "")
)

cat_stats = (
    cat.groupBy("decade", "category")
       .count()
       .withColumn("total_decade", F.sum("count").over(Window.partitionBy("decade")))
       .withColumn("category_pct", F.round(100 * F.col("count") / F.col("total_decade"), 2))
)

# Pick the top category (more than one if tied)
w_cat = Window.partitionBy("decade").orderBy(F.desc("count"))
cat_stats = cat_stats.withColumn("cat_rank", F.rank().over(w_cat))
top_cat = cat_stats.filter(F.col("cat_rank") == 1)

# 3. Explode genres per decade
genre = (
    df
    .withColumn("genre", F.explode(F.split(F.col("Genres"), ";")))
    .withColumn("genre", F.trim(F.lower(F.col("genre"))))
    .filter(F.col("genre") != "")
)

genre_stats = (
    genre.groupBy("decade", "genre")
         .count()
         .withColumn("total_decade", F.sum("count").over(Window.partitionBy("decade")))
         .withColumn("genre_pct", F.round(100 * F.col("count") / F.col("total_decade"), 2))
)

w_genre = Window.partitionBy("decade").orderBy(F.desc("count"))
genre_stats = genre_stats.withColumn("genre_rank", F.rank().over(w_genre))
top_genre = genre_stats.filter(F.col("genre_rank") == 1)

# 4. Join the results per decade (ties give more than one row)
final = (
    top_cat.join(top_genre, "decade")
           .select(
               "decade",
               "category", "category_pct",
               "genre", "genre_pct"
           )
           .orderBy("decade")
)

display(final)


decade,category,category_pct,genre,genre_pct
1990,single-player,18.52,action,60.0
2000,single-player,39.17,action,27.21
2010,single-player,25.13,indie,25.46
2020,single-player,32.28,indie,23.82


# Analysis 5: Price vs. demand by genre

Pricing has a big effect on how a game does, but not the same effect in every genre. Some players will happily pay more for an AAA action game or an RPG, while players of smaller or educational titles are more price sensitive. Knowing how the number of players changes with price helps publishers and developers position their games, plan discounts, and choose between more revenue or a wider audience.

Steam doesn't publish sales numbers, so owners are estimated as the midpoint of the `EstimatedOwners` range. The measure is the correlation between log(price) and log(owners), overall and per genre. It's a rough proxy for price elasticity rather than a fitted elasticity coefficient, and free games drop out because log(0) is undefined.

Result: overall the correlation is positive (0.35), so more expensive games tend to have more owners, most likely because bigger, better-known games also cost more. Mainstream genres such as action, strategy, racing and RPG show the strongest positive relationship (about 0.22–0.24), which suggests players there link price with quality and brand. Creative software (animation & modeling, design & illustration, photo editing) and utilities are the only group where a lower price goes with more owners, and even there the effect is weak (about -0.09 at most). Education and casual sit close to zero. So different genres probably need different pricing: promotions and discounts for niche and specialised titles, premium pricing for the popular genres.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# 1) Map owner-ranges to midpoints
def owners_midpoint(range_str):
    try:
        lo, hi = [float(x.replace(",", "")) for x in range_str.split(" - ")]
        return (lo + hi) / 2.0
    except:
        return None

mid_udf = F.udf(owners_midpoint, DoubleType())

df_elastic = (
    raw_df
      .withColumn("PriceD", F.col("Price").cast("double"))
      .withColumn("OwnersMid", mid_udf(F.col("EstimatedOwners")))
      .filter(F.col("PriceD").isNotNull() & F.col("OwnersMid").isNotNull())
      .withColumn("log_price",  F.log(F.col("PriceD")))
      .withColumn("log_owners", F.log(F.col("OwnersMid")))
)

# 2) Overall log-log correlation (rough elasticity proxy)
elasticity = df_elastic.stat.corr("log_price", "log_owners")
print(f"Estimated price elasticity of demand = {elasticity:.2f}")

# 3) Same per genre, dropping nulls
genre_elastic = (
    df_elastic
      .withColumn("genre", F.explode(F.split(F.col("Genres"), ";")))
      .groupBy("genre")
      .agg(F.expr("corr(log_price, log_owners)").alias("elasticity"))
      .filter(F.col("elasticity").isNotNull())       # drop null rows
      .orderBy(F.col("elasticity").asc())            # most elastic (most negative) first
)

display(genre_elastic)


Estimated price elasticity of demand = 0.35


genre,elasticity
Animation & Modeling,-0.09203396557567635
Design & Illustration,-0.08617033229096752
Violent,-0.08579036976983492
Photo Editing,-0.08438560574986002
Gore,-0.08130864213588751
Software Training,-0.045302710236356014
Utilities,-0.022761612894481347
Nudity,-0.013513046066279963
Game Development,0.002781840002115696
Video Production,0.030046625179143114
